In [2]:
import networkx as nx
import matplotlib.pyplot as plt
import random
import numpy as np
import pandas as pd
from scipy.linalg import eigvals


In [3]:

G = nx.read_graphml("data/test_graph.graphml",node_type=int)

In [5]:
node_data = dict(G.nodes(data=True))

# Create a DataFrame where the index is the node IDs and columns are the attributes
df_nodes = pd.DataFrame.from_dict(node_data, orient='index')


In [6]:
df_nodes[df_nodes['label']==1]

,label
338951,1
94549,1
196183,1
313330,1
212425,1
...,...
161680,1
41583,1
256635,1
9678,1


In [30]:
egonet = nx.ego_graph(G, 212425, radius=1, center=True, undirected=True)


In [31]:
adj_matrix = nx.to_numpy_array(egonet, weight='weight', nodelist=sorted(egonet.nodes()))
A = nx.to_numpy_array(egonet, weight='weight')  # 'weight' ensures that edge weights are used
L = nx.laplacian_matrix(egonet, weight='weight').toarray()
eigenvalues_A = eigvals(A)
eigenvalues_L = eigvals(L)

print("Eigenvalues of the Weighted Adjacency Matrix:")
print(np.sort(eigenvalues_A))

print("\nEigenvalues of the Weighted Laplacian Matrix:")
print(np.sort(eigenvalues_L))

real_parts = eigenvalues_L.real

# Extract the imaginary part of each number
imaginary_parts = eigenvalues_L.imag

Eigenvalues of the Weighted Adjacency Matrix:
[0.+0.j 0.+0.j 0.+0.j 0.+0.j]

Eigenvalues of the Weighted Laplacian Matrix:
[0.+0.j 1.+0.j 1.+0.j 2.+0.j]


In [32]:
# no weight
adjacency_matrix = nx.to_numpy_array(egonet)
degree_matrix = np.diag([egonet.out_degree(n) for n in egonet.nodes()])
laplacian_matrix = degree_matrix - adjacency_matrix
weight_attr = 'total'


laplacian_eigenvalues = np.linalg.eigvals(laplacian_matrix)
sorted_laplacian_eigenvalues = np.sort(laplacian_eigenvalues)



In [33]:
sorted_laplacian_eigenvalues

array([0., 1., 1., 2.])

In [34]:
# Get the weighted adjacency matrix of the graph
weighted_adjacency_matrix = nx.to_numpy_array(egonet, weight=weight_attr)

# Compute the weighted out-degree matrix (sum of outgoing edge weights for each node)
weighted_degree_matrix = np.diag([sum(data.get(weight_attr, 1) for _, _, data in egonet.out_edges(n, data=True)) for n in egonet.nodes()])
weighted_laplacian_matrix = weighted_degree_matrix - weighted_adjacency_matrix
weighted_laplacian_eigenvalues = np.linalg.eigvals(weighted_laplacian_matrix)
sorted_weighted_laplacian_eigenvalues = np.sort(weighted_laplacian_eigenvalues) 

In [35]:
sorted_weighted_laplacian_eigenvalues

array([    0.,   457.,  7195., 98808.])

In [38]:
weighted_adjacency_matrix = nx.to_numpy_array(egonet, weight=weight_attr)
# Compute the weighted degree matrix (sum of outgoing edge weights for each node)
weighted_degree_matrix = np.diag([sum(data.get(weight_attr, 1) for _, _, data in egonet.out_edges(n, data=True)) for n in egonet.nodes()])

# Avoid division by zero in the degree matrix (isolated nodes)
with np.errstate(divide='ignore'):
    inv_sqrt_degree_matrix = np.diag(1.0 / np.sqrt(np.diag(weighted_degree_matrix)))

# Replace infinities with zeros (due to isolated nodes)
inv_sqrt_degree_matrix[np.isinf(inv_sqrt_degree_matrix)] = 0

# Compute the symmetrically normalized Laplacian: L_sym = I - D^(-1/2) * A * D^(-1/2)
identity_matrix = np.eye(weighted_adjacency_matrix.shape[0])
normalized_laplacian_matrix = identity_matrix - inv_sqrt_degree_matrix @ weighted_adjacency_matrix @ inv_sqrt_degree_matrix

# Calculate eigenvalues of the symmetrically normalized Laplacian matrix
sym_norm_laplacian_eigenvalues = np.linalg.eigvals(normalized_laplacian_matrix)

# Sort the eigenvalues in ascending order and return the real part
sorted_sym_norm_laplacian_eigenvalues = np.sort(sym_norm_laplacian_eigenvalues)

In [39]:
sorted_sym_norm_laplacian_eigenvalues

array([1., 1., 1., 1.])